In [1]:
# Notebook: 09_routing_operational
# Operational review-routing analysis (reviewer R7/R8), WITHOUT retraining. Uses nb01 artifacts.
#   - item-level budget-recall (risk-coverage) curves
#   - entity-level audit: rank taxonomy (depth-4) groups, review whole groups until budget
#   - recall@budget table (5/10/20%) and percentile-choice sensitivity for the group score.
import os, csv
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"]="0.2"; plt.rcParams["axes.linewidth"]=0.8; plt.rcParams["font.family"]="DejaVu Sans"
GREYS=["#111111","#555555","#888888","#bbbbbb","#dddddd"]
FIG=os.path.join("..","results","figures"); TAB=os.path.join("..","results","tables")
os.makedirs(FIG,exist_ok=True); os.makedirs(TAB,exist_ok=True)
DATA_DIR=os.path.join("..","data")
meta=pd.read_parquet(os.path.join(DATA_DIR,"item_meta.parquet"))
pi=np.load(os.path.join(DATA_DIR,"pi_memmap.npy"),mmap_mode="r"); N,K=pi.shape
classes=np.load(os.path.join(DATA_DIR,"label_classes.npy"))
noisy=meta["noisy_id"].values; cleanid=meta["clean_id"].values; mis=meta["is_misregistered"].values
p_noisy=meta["p_noisy"].values; entv=meta["entropy"].values; valid=cleanid>=0

def budget_recall(score, is_err, budgets):
    order=np.argsort(-score); err=is_err[order].astype(float)
    cum=np.cumsum(err); tot=max(err.sum(),1); n=len(score)
    return np.array([cum[min(int(round(b*n)),n)-1]/tot for b in budgets])
budgets=np.linspace(0.01,1.0,100); ie=mis[valid].astype(bool)
curves={"1 - p_noisy": budget_recall((1-p_noisy)[valid], ie, budgets),
        "entropy":     budget_recall(entv[valid], ie, budgets),
        "random":      budget_recall(np.random.default_rng(0).random(valid.sum()), ie, budgets)}
fig,ax=plt.subplots(1,2,figsize=(9,3.5))
sty={"1 - p_noisy":dict(color=GREYS[0],ls="-"),"entropy":dict(color=GREYS[1],ls="--"),"random":dict(color=GREYS[3],ls=":")}
for k,c in curves.items(): ax[0].plot(budgets*100,c,label=k,**sty[k])
ax[0].plot([0,100],[0,1],color=GREYS[2],lw=0.8,ls=(0,(1,3)))
ax[0].set_xlabel("item review budget (%)"); ax[0].set_ylabel("recall of misregistrations")
ax[0].legend(frameon=False,fontsize=8); ax[0].set_title("item-level",fontsize=9)

p=os.path.join(DATA_DIR,"category_mapping.csv")
with open(p,encoding="utf-8",errors="replace") as f: hd=[f.readline() for _ in range(6)]
try: sep=csv.Sniffer().sniff("".join(hd),delimiters=[",","\t",";","|"]).delimiter
except Exception: sep="\t"
cmap=pd.read_csv(p,sep=sep,engine="python"); lab2name=dict(zip(cmap.category_label,cmap.category_name))
leaf_parts=[([s.strip() for s in lab2name.get(int(classes[d]),"").split(">")]
             if lab2name.get(int(classes[d]),"") else []) for d in range(K)]
DEPTH=4
leaf_parent=np.array([" > ".join(pp[:DEPTH]) if len(pp)>=DEPTH else None for pp in leaf_parts],dtype=object)
ip=np.array([leaf_parent[d] for d in noisy],dtype=object)
ser=pd.Series(ip); sizes=ser.value_counts()
keep=[k for k,v in sizes.items() if k is not None and 30<=v<=50000]
EPS=1e-12
def ent(x): x=np.clip(x,EPS,1); return -np.sum(x*np.log(x))
def jsd(q,pp): q=np.clip(q,EPS,1);pp=np.clip(pp,EPS,1);m=0.5*(q+pp); return float(0.5*np.sum(q*np.log2(q/m))+0.5*np.sum(pp*np.log2(pp/m)))
S_of={}
for d in range(K):
    pk=leaf_parent[d]
    if pk is not None: S_of.setdefault(pk,[]).append(d)
rows=[]
for gk in keep:
    gi=np.where(ip==gk)[0]; vi=valid[gi]
    if not vi.any(): continue
    S=np.array(S_of[gk])
    pbar=np.asarray(pi[gi][:,S],dtype=np.float64); pbar=(pbar/pbar.sum(1,keepdims=True)).mean(0)
    pos={c:i for i,c in enumerate(S)}
    q=np.bincount([pos[c] for c in noisy[gi]],minlength=len(S)).astype(float); q/=q.sum()
    C=jsd(q,pbar); r=np.clip(pbar-q,0,None); kap=0.0 if r.sum()<=EPS else 1-ent(r/r.sum())/np.log(len(S))
    s_ind=1.0-p_noisy[gi]
    rows.append(dict(group=gk, n=len(gi), n_err=int(mis[gi][vi].sum()), n_valid=int(vi.sum()),
                     p90=float(np.quantile(s_ind,0.9)), Cproj_kappa=C*kap))
G=pd.DataFrame(rows); TOTerr=G.n_err.sum()
def group_audit_curve(scorecol):
    o=G.sort_values(scorecol,ascending=False)
    return np.cumsum(o.n.values)/N, np.cumsum(o.n_err.values)/max(TOTerr,1)
for scol,st in [("p90",dict(color=GREYS[0],ls="-")),("Cproj_kappa",dict(color=GREYS[1],ls="--"))]:
    fr,rc=group_audit_curve(scol); ax[1].plot(fr*100,rc,label=scol,**st)
ax[1].plot([0,100],[0,1],color=GREYS[2],lw=0.8,ls=(0,(1,3)),label="proportional")
ax[1].set_xlabel("items reviewed (%) via top groups"); ax[1].set_ylabel("recall of misregistrations")
ax[1].legend(frameon=False,fontsize=8); ax[1].set_title("entity-level audit",fontsize=9)
fig.tight_layout()
for e in ("png","pdf"): fig.savefig(os.path.join(FIG,f"f09_routing.{e}"),dpi=600,bbox_inches="tight")
plt.close(fig)
tab=[]
for b in (0.05,0.10,0.20):
    row={"budget":b}
    for k,c in curves.items(): row[f"item_{k}"]=round(float(c[np.argmin(np.abs(budgets-b))]),3)
    tab.append(row)
rb=pd.DataFrame(tab); rb.to_csv(os.path.join(TAB,"t09_recall_at_budget.csv"),index=False)
print("recall@budget (item-level):"); print(rb.to_string(index=False))
def auroc_hilo(df,col):
    lo,hi=df.noise.quantile(1/3),df.noise.quantile(2/3)
    sub=df[(df.noise<=lo)|(df.noise>=hi)]; y=(sub.noise>=hi).astype(int).values
    if len(np.unique(y))<2: return np.nan
    a=roc_auc_score(y,sub[col].values); return max(a,1-a)
G["noise"]=G.n_err/np.maximum(G.n_valid,1)
sens=[]
for qv in (0.50,0.75,0.90,0.95):
    col=f"pq{int(qv*100)}"
    G[col]=[float(np.quantile(1.0-p_noisy[np.where(ip==gk)[0]],qv)) for gk in G.group]
    sens.append(dict(percentile=qv, spearman=round(spearmanr(G[col],G.noise)[0],3), auroc=round(auroc_hilo(G,col),3)))
sd=pd.DataFrame(sens); sd.to_csv(os.path.join(TAB,"t09_percentile_sensitivity.csv"),index=False)
print("\npercentile sensitivity:"); print(sd.to_string(index=False))
print("\nsaved f09_routing.{png,pdf}, t09_recall_at_budget.csv, t09_percentile_sensitivity.csv")

C:\Users\miy\AppData\Local\Temp\ipykernel_31892\3091957172.py:61: RuntimeWarning: invalid value encountered in divide
  pbar=np.asarray(pi[gi][:,S],dtype=np.float64); pbar=(pbar/pbar.sum(1,keepdims=True)).mean(0)


recall@budget (item-level):
 budget  item_1 - p_noisy  item_entropy  item_random
   0.05             0.056         0.052        0.049
   0.10             0.108         0.107        0.099
   0.20             0.213         0.212        0.200

percentile sensitivity:
 percentile  spearman  auroc
       0.50     0.241  0.661
       0.75     0.285  0.682
       0.90     0.313  0.693
       0.95     0.312  0.688

saved f09_routing.{png,pdf}, t09_recall_at_budget.csv, t09_percentile_sensitivity.csv
